# XGBoost và LightGBM trên Kaggle

Notebook này gọi pipeline chung `scripts/run_models.py`. Cần attach Dataset code của repo, Dataset Parquet features, và competition dataset để lấy `sample_submission.csv`.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working')

# Chỉnh ba đường dẫn này nếu phần dò tự động tìm nhiều Dataset hoặc không tìm thấy.
PROJECT_ROOT = None
FEATURES_ROOT = None
SAMPLE_SUBMISSION = None

STAGE = 'validation'  # đổi thành 'test' để xuất submission
MODELS = ['xgboost', 'lightgbm']
CV_FOLDS = 3
LOG_PERIOD = 100  # in mỗi 100 vòng boosting

def one_or_none(values, label):
    values = list(dict.fromkeys(values))
    if len(values) > 1:
        raise RuntimeError(f'Multiple {label} candidates found; set its path explicitly in this cell: {values}')
    return values[0] if values else None

if PROJECT_ROOT is None:
    PROJECT_ROOT = one_or_none(
        (path.parent.parent for path in INPUT_ROOT.rglob('run_models.py') if path.parent.name == 'scripts'),
        'project roots',
    )
if FEATURES_ROOT is None:
    feature_files = [path for path in INPUT_ROOT.rglob('train.parquet')
                     if path.parent.name in {'tuning', 'validation', 'test'}
                     and (path.parent / 'features.parquet').is_file()]
    FEATURES_ROOT = one_or_none((path.parent.parent for path in feature_files), 'features roots')
if SAMPLE_SUBMISSION is None:
    SAMPLE_SUBMISSION = one_or_none(INPUT_ROOT.rglob('sample_submission.csv'), 'sample submissions')

if PROJECT_ROOT is None or FEATURES_ROOT is None:
    raise FileNotFoundError('Attach the code Dataset and prepared features Dataset, or set PROJECT_ROOT / FEATURES_ROOT above.')
PROJECT_ROOT, FEATURES_ROOT = Path(PROJECT_ROOT), Path(FEATURES_ROOT)
SAMPLE_SUBMISSION = Path(SAMPLE_SUBMISSION) if SAMPLE_SUBMISSION else None
print('Project:', PROJECT_ROOT)
print('Features:', FEATURES_ROOT)
print('Sample submission:', SAMPLE_SUBMISSION)


## Kiểm tra thư viện

Kaggle thường đã có XGBoost và LightGBM. Cell dưới chỉ cài nếu thiếu; nếu notebook chạy khi Internet tắt, hãy dùng môi trường có sẵn hai package.

In [ ]:
packages = {'scikit-learn': 'sklearn', 'xgboost': 'xgboost', 'lightgbm': 'lightgbm'}
missing = [package for package, module in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    requirements = ['scikit-learn>=1.5,<2', 'xgboost>=3.1,<4', 'lightgbm>=4.6,<5']
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *requirements])
print('xgboost:', importlib.util.find_spec('xgboost') is not None)
print('lightgbm:', importlib.util.find_spec('lightgbm') is not None)


## Chạy huấn luyện và đánh giá

Stage `validation` dùng nhãn validation chỉ để chấm sau khi chọn số vòng cây từ CV lịch sử. Stage `test` ghi hai file submission trong `/kaggle/working/outputs/`.

In [ ]:
script = PROJECT_ROOT / 'scripts' / 'run_models.py'
config = PROJECT_ROOT / 'configs' / 'models.json'
output_dir = WORK_ROOT / 'outputs'
command = [sys.executable, str(script),
           '--features-root', str(FEATURES_ROOT),
           '--stage', STAGE,
           '--config', str(config),
           '--output-dir', str(output_dir),
           '--cv-folds', str(CV_FOLDS),
           '--log-period', str(LOG_PERIOD),
           '--models', *MODELS]
if STAGE == 'test':
    if SAMPLE_SUBMISSION is None:
        raise FileNotFoundError('Attach the competition dataset or set SAMPLE_SUBMISSION above.')
    command.extend(['--sample-submission', str(SAMPLE_SUBMISSION)])
print('Running:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
print('Output files:')
for path in sorted(output_dir.rglob('*')):
    if path.is_file():
        print(path, f'({path.stat().st_size:,} bytes)')
